In [15]:
!cd /home/ilya-trushkin/Desktop/digital_signal_processing/

# 2. Активируй виртуальное окружение
!source venv/bin/activate

# 3. Установи пакет (теперь pip поймет, что ставить нужно локально в venv)
!pip install scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg
from sklearn.decomposition import FastICA

np.random.seed(42)
N_CHANNELS = 10
N_TIMEPOINTS = 500
TIME_AXIS = np.arange(N_TIMEPOINTS)
AMP_1, FREQ_1 = 2.0, 0.05
AMP_2, FREQ_2 = 1.0, 0.10
PLOT_LIMIT = 100
VERT_OFFSET = 2.5

spatial_1 = np.sin(np.linspace(0, 2*np.pi, N_CHANNELS))
spatial_2 = np.cos(np.linspace(0, 2*np.pi, N_CHANNELS))
temporal_1 = AMP_1 * np.sin(2*np.pi*FREQ_1*TIME_AXIS)
temporal_2 = AMP_2 * np.sin(2*np.pi*FREQ_2*TIME_AXIS)

def generate_data(noise_level):
    clean = np.outer(spatial_1, temporal_1) + np.outer(spatial_2, temporal_2)
    noise = np.random.randn(N_CHANNELS, N_TIMEPOINTS) * noise_level
    return clean + noise, clean, noise

def normalize_component(weights, temporal_pc, ref_signal=None, target_amp=None):
    scale = np.max(np.abs(weights))
    w_norm, t_norm = weights/scale, temporal_pc*scale
    if ref_signal is not None:
        if np.corrcoef(t_norm, ref_signal)[0,1] < 0:
            w_norm, t_norm = -w_norm, -t_norm
    if target_amp is not None and np.max(np.abs(t_norm)) > 0:
        t_norm = t_norm / np.max(np.abs(t_norm)) * target_amp
    return w_norm, t_norm

def plot_results(noisy_data, components, spatial_weights, titles, plot_limit=100):
    n_chan = noisy_data.shape[0]
    fig, ax = plt.subplots(figsize=(8,3))
    for ch in range(min(5, n_chan)):
        ax.plot(TIME_AXIS[:plot_limit], noisy_data[ch,:plot_limit] + ch*VERT_OFFSET, lw=0.8, alpha=0.7)
    ax.set_xlabel('Время'); ax.set_ylabel('Канал'); ax.set_title('Исходные данные'); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    fig, ax = plt.subplots(figsize=(6,4))
    for w, lbl in zip(spatial_weights[:3], titles[:3]):
        ax.plot(np.arange(len(w)), w, marker='o', ms=3, label=lbl)
    ax.set_xlabel('Канал'); ax.set_ylabel('Вес'); ax.set_title('Пространственные паттерны'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    fig, ax = plt.subplots(figsize=(8,4))
    for t, lbl in zip(components[:3], titles[:3]):
        ax.plot(TIME_AXIS[:plot_limit], t[:plot_limit], label=lbl, lw=1.5)
    ax.plot(TIME_AXIS[:plot_limit], temporal_1[:plot_limit], 'k--', label='Источник 1', alpha=0.7)
    ax.plot(TIME_AXIS[:plot_limit], temporal_2[:plot_limit], 'r--', label='Источник 2', alpha=0.7)
    ax.set_xlabel('Время'); ax.set_ylabel('Амплитуда'); ax.set_title('Временные компоненты'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
NOISE_1 = 0.3
noisy_1, clean_1, noise_1 = generate_data(NOISE_1)

data_centered_1 = noisy_1 - noisy_1.mean(axis=1, keepdims=True)

cov_matrix_1 = data_centered_1 @ data_centered_1.T / (N_TIMEPOINTS - 1)

eigenvalues_1, eigenvectors_1 = np.linalg.eigh(cov_matrix_1)

sorted_idx_1 = np.argsort(eigenvalues_1)[::-1]
eigenvalues_1 = eigenvalues_1[sorted_idx_1]
eigenvectors_1 = eigenvectors_1[:, sorted_idx_1]

principal_components_1 = eigenvectors_1.T @ data_centered_1

w1,t1 = normalize_component(eigenvectors_1[:,0], principal_components_1[0], temporal_1, AMP_1)
w2,t2 = normalize_component(eigenvectors_1[:,1], principal_components_1[1], temporal_2, AMP_2)
w3,t3 = normalize_component(eigenvectors_1[:,2], principal_components_1[2])
plot_results(noisy_1, [t1,t2,t3], [w1,w2,w3], ['PCA1','PCA2','PCA3'])

# Матрица корреляции каналов
corr_matrix = np.corrcoef(noisy_1)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_title('Матрица корреляции каналов')
ax.set_xlabel('Канал'); ax.set_ylabel('Канал')
plt.tight_layout(); plt.show()

# Объяснённая дисперсия
explained_var = eigenvalues_1 / eigenvalues_1.sum()
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(np.arange(1, N_CHANNELS+1), explained_var * 100, color='steelblue', edgecolor='white')
ax.plot(np.arange(1, N_CHANNELS+1), np.cumsum(explained_var) * 100, 'ro-', label='Накопленная')
ax.set_xlabel('Компонента'); ax.set_ylabel('Дисперсия (%)')
ax.set_title('Объяснённая дисперсия (PCA)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

corr1_pca = np.corrcoef(t1, temporal_1)[0,1]
corr2_pca = np.corrcoef(t2, temporal_2)[0,1]
print(f"PCA: corr1 = {corr1_pca:.4f}, corr2 = {corr2_pca:.4f}")
print(f"Объяснённая дисперсия PC1={explained_var[0]*100:.1f}%, PC2={explained_var[1]*100:.1f}%, PC3={explained_var[2]*100:.1f}%")

In [ ]:
NOISE_2 = 0.8
noisy_2, clean_2, noise_2 = generate_data(NOISE_2)

data_c_2 = noisy_2 - noisy_2.mean(axis=1, keepdims=True)
noise_c_2 = noise_2 - noise_2.mean(axis=1, keepdims=True)

cov_sig_2 = data_c_2 @ data_c_2.T / (N_TIMEPOINTS - 1)
cov_noise_2 = noise_c_2 @ noise_c_2.T / (N_TIMEPOINTS - 1)

reg = 1e-8 * np.trace(cov_noise_2) / N_CHANNELS
cov_noise_reg_2 = cov_noise_2 + reg * np.eye(N_CHANNELS)

evals_2, evecs_2 = scipy.linalg.eigh(cov_sig_2, cov_noise_reg_2)

sorted_idx_2 = np.argsort(evals_2)[::-1]
evals_2 = evals_2[sorted_idx_2]
evecs_2 = evecs_2[:, sorted_idx_2]
comp_ts_2 = evecs_2.T @ data_c_2

def compute_forward(w, C_sig, C_noise):
    denom = w.T @ C_noise @ w
    if np.abs(denom) < 1e-12:
        return np.zeros_like(w)
    return (C_sig @ w) / denom

C_sig_diff_2 = cov_sig_2 - cov_noise_2
f1 = compute_forward(evecs_2[:,0], C_sig_diff_2, cov_noise_reg_2)
f2 = compute_forward(evecs_2[:,1], C_sig_diff_2, cov_noise_reg_2)
f3 = compute_forward(evecs_2[:,2], C_sig_diff_2, cov_noise_reg_2)

w1g,t1g = normalize_component(f1, comp_ts_2[0], temporal_1, AMP_1)
w2g,t2g = normalize_component(f2, comp_ts_2[1], temporal_2, AMP_2)
w3g,t3g = normalize_component(f3, comp_ts_2[2])
plot_results(noisy_2, [t1g,t2g,t3g], [w1g,w2g,w3g], ['GED1','GED2','GED3'])

# Спектр обобщённых собственных значений
evals_norm = evals_2 / evals_2[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(np.arange(1, N_CHANNELS+1), evals_norm, color='darkorange', edgecolor='white')
axes[0].set_xlabel('Компонента'); axes[0].set_ylabel('λ / λmax')
axes[0].set_title('Спектр обобщённых собственных значений (GED)')
axes[0].grid(alpha=0.3)
for lbl, fw in zip(['GED1','GED2','GED3'], [w1g, w2g, w3g]):
    axes[1].plot(np.arange(N_CHANNELS), fw, marker='o', ms=4, label=lbl)
axes[1].set_xlabel('Канал'); axes[1].set_ylabel('Вес')
axes[1].set_title('Пространственные паттерны GED')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Временные ряды с истинными источниками
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(TIME_AXIS[:PLOT_LIMIT], t1g[:PLOT_LIMIT], label='GED1', lw=1.5)
ax.plot(TIME_AXIS[:PLOT_LIMIT], t2g[:PLOT_LIMIT], label='GED2', lw=1.5)
ax.plot(TIME_AXIS[:PLOT_LIMIT], t3g[:PLOT_LIMIT], label='GED3', lw=1.5)
ax.plot(TIME_AXIS[:PLOT_LIMIT], temporal_1[:PLOT_LIMIT], 'k--', label='Источник 1', alpha=0.7)
ax.plot(TIME_AXIS[:PLOT_LIMIT], temporal_2[:PLOT_LIMIT], 'r--', label='Источник 2', alpha=0.7)
ax.set_xlabel('Время'); ax.set_ylabel('Амплитуда')
ax.set_title('GED: временные ряды компонент vs истинные источники')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

corr1_ged = np.corrcoef(t1g, temporal_1)[0,1]
corr2_ged = np.corrcoef(t2g, temporal_2)[0,1]
print(f"GED: corr1 = {corr1_ged:.4f}, corr2 = {corr2_ged:.4f}")

In [4]:
X_ica_input = noisy_2 - noisy_2.mean(axis=1, keepdims=True)

X_ica = X_ica_input.T

ica = FastICA(n_components=N_CHANNELS, random_state=42, whiten='arbitrary-variance')
sources = ica.fit_transform(X_ica)
mixing = ica.mixing_

sources = sources.T
spatial_maps = mixing.T

energy = np.sum(spatial_maps**2, axis=1)
sorted_idx = np.argsort(energy)[::-1]
spatial_maps = spatial_maps[sorted_idx]
sources = sources[sorted_idx]

corrs_1 = [np.abs(np.corrcoef(sources[i], temporal_1)[0,1]) for i in range(N_CHANNELS)]
corrs_2 = [np.abs(np.corrcoef(sources[i], temporal_2)[0,1]) for i in range(N_CHANNELS)]
idx1 = np.argmax(corrs_1)
idx2 = np.argmax(corrs_2)

w1i,t1i = normalize_component(spatial_maps[idx1], sources[idx1], temporal_1, AMP_1)
w2i,t2i = normalize_component(spatial_maps[idx2], sources[idx2], temporal_2, AMP_2)

# Энергия компонент (нормированная)
energy_norm = energy[np.argsort(energy)[::-1]] / energy.max()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(np.arange(1, N_CHANNELS+1), energy_norm, color='mediumseagreen', edgecolor='white')
axes[0].set_xlabel('Компонента (отсортированы по энергии)')
axes[0].set_ylabel('Энергия / max')
axes[0].set_title('Энергия компонент ICA (нормированная)')
axes[0].grid(alpha=0.3)
for lbl, fw in zip([f'ICA{idx1+1}', f'ICA{idx2+1}', 'ICA1'], [w1i, w2i, spatial_maps[0]]):
    axes[1].plot(np.arange(N_CHANNELS), fw, marker='o', ms=4, label=lbl)
axes[1].set_xlabel('Канал'); axes[1].set_ylabel('Вес')
axes[1].set_title('Пространственные карты ICA')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Временные ряды ICA с истинными источниками
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(TIME_AXIS[:PLOT_LIMIT], t1i[:PLOT_LIMIT], label=f'ICA{idx1+1}', lw=1.5)
ax.plot(TIME_AXIS[:PLOT_LIMIT], t2i[:PLOT_LIMIT], label=f'ICA{idx2+1}', lw=1.5)
ax.plot(TIME_AXIS[:PLOT_LIMIT], sources[0][:PLOT_LIMIT], label='ICA1', lw=1.5, alpha=0.5)
ax.plot(TIME_AXIS[:PLOT_LIMIT], temporal_1[:PLOT_LIMIT], 'k--', label='Источник 1', alpha=0.7)
ax.plot(TIME_AXIS[:PLOT_LIMIT], temporal_2[:PLOT_LIMIT], 'r--', label='Источник 2', alpha=0.7)
ax.set_xlabel('Время'); ax.set_ylabel('Амплитуда')
ax.set_title('ICA: временные ряды компонент vs истинные источники')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

corr1_ica = np.corrcoef(t1i, temporal_1)[0,1]
corr2_ica = np.corrcoef(t2i, temporal_2)[0,1]
print(f"ICA: corr1 = {corr1_ica:.4f}, corr2 = {corr2_ica:.4f}")

NameError: name 'noisy_2' is not defined

In [5]:
print("Метод | corr1  | corr2")
print(f"PCA   | {corr1_pca:.4f} | {corr2_pca:.4f}")
print(f"GED   | {corr1_ged:.4f} | {corr2_ged:.4f}")
print(f"ICA   | {corr1_ica:.4f} | {corr2_ica:.4f}")

Метод | corr1  | corr2


NameError: name 'corr1_pca' is not defined